# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is accessible via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

We'll display the dataset's main name and description after loading.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and IDs. We'll use `@id` references for all entities.

Let's list available `RecordSet` objects and their constituent field and column IDs.

In [ ]:
# List RecordSets and their fields using @id
print("--- Available RecordSets ---")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    record_sets.append(rs["@id"])
    print("  Fields and Columns:")
    for f in rs.get("fields", []):
        field_id = f["@id"]
        column_ids = [c["@id"] for c in f.get("columns", [])]
        print(f"    Field @id: {field_id}, Columns: {column_ids}")
    print()
if not record_sets:
    print('No record sets found in schema.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We use the RecordSet `@id` from above for extraction. Replace `<record_set_id>` with your RecordSet of interest.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df.copy()
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample data:\n{df.head(2)}\n")
if not dataframes:
    print('No records extracted. Check schema for available records.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing and investigation steps, such as filtering numeric records, normalizing, and grouping.

**Note:** Replace example variable values with actual field names/IDs from the DataFrame output above.

In [ ]:
# ----
# Example EDA. 
# Select a DataFrame and fields by @id (replace with your actual RecordSet and field IDs as listed above).
import numpy as np
if record_sets:
    record_set_id = record_sets[0]  # Use the first record set found, or set explicitly
    df = dataframes[record_set_id]
    print(f'Working with DataFrame for RecordSet: {record_set_id}\nColumns: {df.columns.tolist()}')

    # Try to auto-select a likely numeric field (improvement: you may replace with the actual '@id' like 'cr:field/protein_coverage')
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not numeric_field_candidates:
        # Try to infer numeric fields despite all columns being object
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        numeric_field_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]

    print(f"Detected numeric fields: {numeric_field_candidates}")

    if numeric_field_candidates:
        numeric_field = numeric_field_candidates[0]
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records: {numeric_field} > {threshold:.3f}\n", filtered_df.head(5))

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} (z-score):\n", filtered_df[[numeric_field, norm_col]].head())

        # Try a categorical grouping field, e.g. any non-numeric field
        group_field_candidates = [col for col in df.columns if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col])]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            if group_field in df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
                print(f"\nGrouped mean {numeric_field} by {group_field}:")
                print(grouped_df.head())
    else:
        print('No numeric fields were automatically detected. Please specify them manually for analysis.')
else:
    print('No record sets available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib/seaborn.

> _Tip: Adjust field names to match your dataset's `@id` field names as found in the DataFrame output above_.


In [ ]:
# Example: Plot histogram and boxplot for a numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field_candidates:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    numeric_field = numeric_field_candidates[0]

    plt.figure(figsize=(14, 4))
    plt.subplot(1, 2, 1)
    df[numeric_field].hist(bins=30)
    plt.title(f'Histogram of {numeric_field}')

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field])
    plt.title(f'Boxplot of {numeric_field}')

    plt.tight_layout()
    plt.show()

    # If grouping field is found, make a violin or bar plot
    if group_field_candidates:
        group_field = group_field_candidates[0]
        plt.figure(figsize=(7, 4))
        sns.barplot(x=group_field, y=numeric_field, data=df, estimator=np.mean, ci=None)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric fields detected for visualization.')

## 6. Conclusion
We successfully loaded the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

- We explored the available record sets and their field `@id`s.
- We extracted records into pandas DataFrames for flexible analysis.
- We performed basic EDA and visualized example numeric fields, providing a solid foundation for deeper biological or statistical insights.

Continue with domain-driven analyses as required for your application.